In [10]:
import json
import pandas as pd
from collections import defaultdict


with open("label_studio/gold_standard_25_rulings_THOM.json") as f:
    data_thom = json.load(f)

with open("label_studio/gold_standard_25_rulings_JASON.json") as f:
    data_jason = json.load(f)

print(f"thom:   {len(data_thom)} uitspraken")
print(f"jason: {len(data_jason)} uitspraken")

thom:   25 uitspraken
jason: 31 uitspraken


In [11]:
# ── 2. Extraheer spans per ECLI ───────────────────────────────────────────────

def extraheer_spans(data):
    """Extraheert (ecli, start, end, label) tuples uit Label Studio JSON."""
    resultaat = {}
    
    for item in data:
        ecli = item["data"].get("ecli", item["data"].get("text", "")[:50])
        spans = set()
        
        for ann in item.get("annotations", []):
            for result in ann.get("result", []):
                if result.get("type") == "labels":
                    val = result["value"]
                    label = val["labels"][0] if val.get("labels") else None
                    if label in ("BWB", "ECLI", "CELEX", "OEP"):
                        spans.add((val["start"], val["end"], label))
        
        resultaat[ecli] = spans
    
    return resultaat

spans_thom   = extraheer_spans(data_thom)
spans_jason = extraheer_spans(data_jason)

print(f"thom:   {sum(len(v) for v in spans_thom.values())} spans in {len(spans_thom)} uitspraken")
print(f"jason: {sum(len(v) for v in spans_jason.values())} spans in {len(spans_jason)} uitspraken")

thom:   344 spans in 25 uitspraken
jason: 278 spans in 31 uitspraken


In [12]:
# ── 3. Vind overlappende uitspraken ──────────────────────────────────────────

eclis_thom   = set(spans_thom.keys())
eclis_jason = set(spans_jason.keys())
overlap_eclis = eclis_thom & eclis_jason

print(f"Uitspraken in beide: {len(overlap_eclis)}")
print(f"Alleen thom:          {len(eclis_thom - eclis_jason)}")
print(f"Alleen jason:        {len(eclis_jason - eclis_thom)}")

Uitspraken in beide: 25
Alleen thom:          0
Alleen jason:        6


In [13]:
# ── 4. Vergelijk per uitspraak ────────────────────────────────────────────────

vergelijking = []

for ecli in sorted(overlap_eclis):
    s_thom   = spans_thom[ecli]
    s_jason = spans_jason[ecli]
    
    alleen_thom   = s_thom - s_jason
    alleen_jason = s_jason - s_thom
    beiden       = s_thom & s_jason
    
    vergelijking.append({
        "ecli":        ecli,
        "thom":         len(s_thom),
        "jason":       len(s_jason),
        "overeenstemming": len(beiden),
        "alleen_thom":  len(alleen_thom),
        "alleen_jason": len(alleen_jason),
        "jaccard":     len(beiden) / len(s_thom | s_jason) if s_thom | s_jason else 1.0
    })

df_vergelijking = pd.DataFrame(vergelijking)
print(df_vergelijking.to_string(index=False))

                    ecli  thom  jason  overeenstemming  alleen_thom  alleen_jason  jaccard
   ECLI:NL:CRVB:2021:771     5      5                4            1             1 0.666667
 ECLI:NL:GHAMS:2018:2581     8      8                3            5             5 0.230769
 ECLI:NL:GHARL:2021:6207     6      2                0            6             2 0.000000
 ECLI:NL:GHDHA:2018:3446     1      1                1            0             0 1.000000
  ECLI:NL:GHSHE:2018:221     8     10                4            4             6 0.285714
  ECLI:NL:GHSHE:2019:962     2      2                2            0             0 1.000000
 ECLI:NL:GHSHE:2020:1630     1      1                1            0             0 1.000000
 ECLI:NL:GHSHE:2021:1733     4      4                3            1             1 0.600000
   ECLI:NL:PHR:2018:1249    81     66               51           30            15 0.531250
    ECLI:NL:PHR:2021:399    64     31               26           38             5 0.376812

In [14]:
# ── 5. Jaccard similarity overall ────────────────────────────────────────────

alle_thom   = set()
alle_jason = set()

for ecli in overlap_eclis:
    for span in spans_thom[ecli]:
        alle_thom.add((ecli,) + span)
    for span in spans_jason[ecli]:
        alle_jason.add((ecli,) + span)

intersection = alle_thom & alle_jason
union        = alle_thom | alle_jason

jaccard = len(intersection) / len(union) if union else 1.0

print(f"Totaal spans thom:        {len(alle_thom)}")
print(f"Totaal spans jason:      {len(alle_jason)}")
print(f"Overeenstemming:         {len(intersection)}")
print(f"Jaccard similarity:      {jaccard:.3f}")

Totaal spans thom:        344
Totaal spans jason:      278
Overeenstemming:         205
Jaccard similarity:      0.492


In [21]:
 # ── 6. Cohen's Kappa per label ────────────────────────────────────────────────
# Aanpak: per token (character positie) bepalen welk label beide annotators geven

from sklearn.metrics import cohen_kappa_score
import numpy as np

def spans_naar_token_labels(spans, tekst_len, labels=("BWB", "ECLI", "CELEX", "OEP")):
    """Zet spans om naar een array van labels per character positie."""
    tokens = ["O"] * tekst_len
    for start, end, label in sorted(spans, key=lambda x: x[1]-x[0], reverse=True):
        for i in range(start, min(end, tekst_len)):
            tokens[i] = label
    return tokens

# Bereken kappa per uitspraak en overall
alle_labels_thom   = []
alle_labels_jason = []

for ecli in sorted(overlap_eclis):
    # Bepaal tekstlengte
    item_thom = next(i for i in data_thom if i["data"].get("ecli") == ecli)
    tekst_len = len(item_thom["data"].get("text", ""))
    if tekst_len == 0:
        continue
    
    labels_thom   = spans_naar_token_labels(spans_thom[ecli],   tekst_len)
    labels_jason = spans_naar_token_labels(spans_jason[ecli], tekst_len)
    
    alle_labels_thom.extend(labels_thom)
    alle_labels_jason.extend(labels_jason)

kappa = cohen_kappa_score(alle_labels_thom, alle_labels_jason)
print(f"Cohen's Kappa (overall): {kappa:.3f}")

# Per label
for label in ("BWB", "ECLI", "CELEX", "OEP"):
    y_thom   = [1 if l == label else 0 for l in alle_labels_thom]
    y_jason = [1 if l == label else 0 for l in alle_labels_jason]
    if sum(y_thom) + sum(y_jason) > 0:
        k = cohen_kappa_score(y_thom, y_jason)
        print(f"  {label}: kappa = {k:.3f}")

Cohen's Kappa (overall): 0.837
  BWB: kappa = 0.893
  ECLI: kappa = 0.632
  CELEX: kappa = 0.900
  OEP: kappa = 0.266


In [20]:
# ── 7. Toon verschillen per uitspraak ─────────────────────────────────────────

def get_tekst(ecli, data_primair, data_fallback=None):
    """Haalt tekst op, met fallback naar tweede dataset."""
    for data in [data_primair, data_fallback]:
        if data is None:
            continue
        item = next((i for i in data if i["data"].get("ecli") == ecli), None)
        if item:
            # Probeer beide kolomnamen
            tekst = item["data"].get("text") or item["data"].get("tekst")
            if tekst:
                return tekst
    return ""

print("Spans die alleen thom hebt geannoteerd:")
for ecli in sorted(overlap_eclis):
    alleen_thom = spans_thom[ecli] - spans_jason[ecli]
    if alleen_thom:
        tekst = get_tekst(ecli, data_thom)
        print(f"\n{ecli}:")
        for start, end, label in sorted(alleen_thom):
            span = tekst[start:end] if tekst else "?"
            print(f"  [{label}] '{span}'")

print("\n\nSpans die alleen JASON heeft geannoteerd:")
for ecli in sorted(overlap_eclis):
    alleen_jason = spans_jason[ecli] - spans_thom[ecli]
    if alleen_jason:
        tekst = get_tekst(ecli, data_jason)
        print(f"\n{ecli}:")
        for start, end, label in sorted(alleen_jason):
            span = tekst[start:end] if tekst else "?"
            print(f"  [{label}] '{span}'")

Spans die alleen thom hebt geannoteerd:

ECLI:NL:CRVB:2021:771:
  [BWB] 'Participatiewet'

ECLI:NL:GHAMS:2018:2581:
  [BWB] 'artikel 6 van het Europees Verdrag tot bescherming van de rechten van de mens'
  [BWB] 'EVRM'
  [BWB] 'artikel 6, eerste lid, van het Verdrag tot bescherming van de rechten van de mens en de fundamentele vrijheden'
  [BWB] 'artikelen 47, 57, 63 en 420bis van het Wetboek van Strafrecht'
  [BWB] 'artikel 27, eerste lid, of artikel 27a van het Wetboek van Strafrecht'

ECLI:NL:GHARL:2021:6207:
  [BWB] 'Wet natuurbescherming'
  [BWB] 'Wnb'
  [BWB] 'Wnb'
  [BWB] 'Wnb'
  [BWB] 'Wnb'
  [BWB] 'Wnb'

ECLI:NL:GHSHE:2018:221:
  [BWB] 'artikel 222 Rv jo 220 Rv'
  [BWB] 'artikel 220 Rv'
  [ECLI] 'ECLI:NL:HR:1997:ZC2500'
  [ECLI] 'ECLI:NL:HR:1999:ZC2904'

ECLI:NL:GHSHE:2021:1733:
  [BWB] 'art. 279 Sv'

ECLI:NL:PHR:2018:1249:
  [BWB] 'Aw'
  [ECLI] 'C-5/08'
  [ECLI] 'C-393/09'
  [BWB] 'Auteurswet'
  [BWB] 'Auteurswet'
  [ECLI] 'Hoge Raad in het Stokke/H3 Products-arrest van 22 fe